# 👗 Fashion MNIST CNN Classifier

End-to-end training notebook for a Convolutional Neural Network on the Fashion MNIST dataset.
Run all cells top-to-bottom. The final cell saves `fashion_mnist_cnn.keras`.

**Architecture:** 3× Conv2D blocks (32 → 64 → 128 filters) + BatchNorm + Dropout + Dense(256)  
**Target accuracy:** ~91%+ on test set

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print('TensorFlow:', tf.__version__)

In [ ]:
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
print('Train:', x_train.shape, '| Test:', x_test.shape)

x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, -1)
x_test  = np.expand_dims(x_test, -1)

NUM_CLASSES = 10
y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat  = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)
datagen.fit(x_train)

BATCH_SIZE = 64
train_gen = datagen.flow(x_train, y_train_cat, batch_size=BATCH_SIZE, subset='training', seed=42)
val_gen   = datagen.flow(x_train, y_train_cat, batch_size=BATCH_SIZE, subset='validation', seed=42)
print('Augmented train batches:', len(train_gen))

In [ ]:
model = models.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    # Classifier head
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

history = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Loss:     {test_loss:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
model.save('fashion_mnist_cnn.keras')
print('Model saved as fashion_mnist_cnn.keras')

# Download in Colab
try:
    from google.colab import files
    files.download('fashion_mnist_cnn.keras')
    print('Download triggered.')
except ImportError:
    print('Not in Colab — file saved locally.')